# 623 Stride v21: contract-driven five-capacity LSTM sweep

This notebook takes the run ID, five model points, revisions, external-input fields, and training configuration from the checked-in trainer/model contract. Select the multipart JSON manifest and every numbered input part from the Mac in one upload. Colab saves the parts in Google Drive, verifies every size and SHA256, reassembles the exact `<RUN_ID>.colab_input.tar.gz`, safely extracts it, validates `SHA256SUMS`, trains all contract points, and saves the output archive to Drive. An output larger than 90 MiB is downloaded as verified multipart files.

In [ ]:
import hashlib, json, os, pathlib, shutil, subprocess, sys, tarfile
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import torch
from google.colab import userdata
assert torch.cuda.is_available() and 'A100' in torch.cuda.get_device_name(0), f'Select an A100 runtime, observed {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}'
torch.set_float32_matmul_precision('highest')
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
torch.use_deterministic_algorithms(True)
DEVICE_NAME=torch.cuda.get_device_name(0)
REPO='/content/cache_arch'; TOKEN=userdata.get('GITHUB_TOKEN')
assert TOKEN, 'Add GITHUB_TOKEN to Colab Secrets'
ASKPASS='/content/cache_arch_git_askpass.sh'
pathlib.Path(ASKPASS).write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$GITHUB_TOKEN" ;; esac\n')
os.chmod(ASKPASS,0o700)
env=os.environ.copy(); env.update({'GIT_ASKPASS':ASKPASS,'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN':TOKEN})
try:
    if not os.path.isdir(REPO): subprocess.run(['git','clone','https://github.com/Angelawoo572/cache_arch.git',REPO],check=True,env=env)
    else: subprocess.run(['git','-C',REPO,'pull','--ff-only','origin','main'],check=True,env=env)
finally:
    if pathlib.Path(ASKPASS).exists(): pathlib.Path(ASKPASS).unlink()
print(DEVICE_NAME,subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip())

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
SCRIPT=f'{REPO}/formal_NN_training/experiments/623_offline_lstm_stride/python/train_and_offline_infer.py'
CONTRACT_SCRIPT=f'{REPO}/formal_NN_training/experiments/623_offline_lstm_stride/python/model_contract.py'
MODEL_CONTRACT=json.loads(subprocess.check_output([sys.executable,CONTRACT_SCRIPT,'--describe-model-points'],text=True))
TRAINER_CONTRACT=json.loads(subprocess.check_output([sys.executable,SCRIPT,'--describe-model-points'],text=True))
assert MODEL_CONTRACT==TRAINER_CONTRACT, 'trainer and model_contract descriptions differ'
for key in ('run_id','points','training_config','model_revision','decoder_revision','external_input_fields'): assert key in MODEL_CONTRACT,key
RUN_ID=MODEL_CONTRACT['run_id']; TRAINING=MODEL_CONTRACT['training_config']
assert 'v21' in RUN_ID and 'v21' in MODEL_CONTRACT['model_revision'] and 'v21' in MODEL_CONTRACT['decoder_revision'],MODEL_CONTRACT
DRIVE_ROOT=f'/content/drive/MyDrive/cache_prefetch_623_stride/{RUN_ID}'
INPUT_DIR=f'/content/{RUN_ID}_colab_input'; OUTPUT_ROOT=f'{DRIVE_ROOT}/colab_output'
INPUT_PARTS_DIR=f'{DRIVE_ROOT}/input_transfer_parts'; os.makedirs(DRIVE_ROOT,exist_ok=True)
COMMON=f'{REPO}/formal_NN_training/common'; sys.path.insert(0,COMMON)
import split_colab_archive as transfer
assert transfer.MAX_PART_BYTES==90*1024*1024
expected_archive_name=f'{RUN_ID}.colab_input.tar.gz'
print(f'Select {expected_archive_name}.parts.json and every numbered part in the same upload.')
uploaded=files.upload(); selected_names=set(uploaded)
assert uploaded and all(pathlib.Path(name).name==name and '\\' not in name for name in uploaded),sorted(uploaded)
manifest_names=[name for name in uploaded if name.endswith('.parts.json')]
assert len(manifest_names)==1,f'Select exactly one multipart manifest plus all parts; observed {sorted(uploaded)}'
if pathlib.Path(INPUT_PARTS_DIR).exists(): shutil.rmtree(INPUT_PARTS_DIR)
pathlib.Path(INPUT_PARTS_DIR).mkdir(parents=True)
for upload_name,payload in uploaded.items(): pathlib.Path(INPUT_PARTS_DIR,upload_name).write_bytes(payload)
manifest_path=pathlib.Path(INPUT_PARTS_DIR,manifest_names[0])
PART_MANIFEST=transfer.load_manifest(manifest_path)
assert PART_MANIFEST['archive']['name']==expected_archive_name,PART_MANIFEST['archive']
assert manifest_path.name==f'{expected_archive_name}.parts.json',manifest_path.name
expected_upload_names={manifest_path.name}|{part['name'] for part in PART_MANIFEST['parts']}
assert selected_names==expected_upload_names,(sorted(selected_names),sorted(expected_upload_names))
del uploaded
transfer.validate_parts(PART_MANIFEST,INPUT_PARTS_DIR)
archive_path=pathlib.Path(DRIVE_ROOT,expected_archive_name)
transfer.reassemble_archive(manifest_path,INPUT_PARTS_DIR,archive_path,overwrite=True)
assert transfer.sha256_file(archive_path)==PART_MANIFEST['archive']['sha256']
if pathlib.Path(INPUT_DIR).exists(): shutil.rmtree(INPUT_DIR)
transfer.safe_extract_tar_gz(archive_path,INPUT_DIR)
SHA256SUMS_VERIFIED=transfer.validate_sha256sums(INPUT_DIR)
print('verified multipart input',archive_path,PART_MANIFEST['archive']['size_bytes'],'bytes',len(SHA256SUMS_VERIFIED),'payload hashes')

In [ ]:
TRACE=MODEL_CONTRACT['trace']; POLICY=MODEL_CONTRACT['policy']; ROLES=('train','guard','eval')
INPUTS={role:{'stream':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_stream.csv.gz','candidates':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_candidates.csv.gz'} for role in ROLES}
for role,items in INPUTS.items():
    for path in items.values(): assert os.path.isfile(path),path
collection_manifest=json.loads(pathlib.Path(f'{INPUT_DIR}/collection_manifest.json').read_text())
external_fields=MODEL_CONTRACT['external_input_fields']
expected_input={'status':'PASS','source_decision_effective_external_input':external_fields,'same_external_input_contract':True,'training_inference_input_encoder_identical':True,'normal_policy_outputs_used_as_model_inputs':False,'normal_policy_candidates_used_as_model_inputs':False,'normal_policy_private_state_used_as_model_inputs':False,'normal_policy_outputs_used_as_training_targets':True}
bad={k:(collection_manifest.get(k),v) for k,v in expected_input.items() if collection_manifest.get(k)!=v}; assert not bad,bad
assert collection_manifest['training_runtime_fields']==external_fields==collection_manifest['inference_runtime_fields']
assert MODEL_CONTRACT.get('neural_role')=='standalone_direct_action_prefetcher'
assert MODEL_CONTRACT.get('normal_policy_outputs_used_as_model_inputs') is False
assert MODEL_CONTRACT.get('normal_policy_candidates_used_as_model_inputs') is False
assert MODEL_CONTRACT.get('normal_policy_private_state_used_as_model_inputs') is False
print('input contract PASS',TRACE,POLICY,external_fields)

In [ ]:
LOCAL_OUTPUT=f'/content/{RUN_ID}_colab_output'
if os.path.isdir(LOCAL_OUTPUT): shutil.rmtree(LOCAL_OUTPUT)
os.makedirs(LOCAL_OUTPUT)
POINTS=sorted(MODEL_CONTRACT['points'],key=lambda point:point['model_size'])
assert [point['model_size'] for point in POINTS]==[8,16,32,64,128],POINTS
assert len({point['model_tag'] for point in POINTS})==len(POINTS) and len({point['architecture_pair_id'] for point in POINTS})==len(POINTS)
assert all(point['model_family']=='lstm' for point in POINTS)
source_paths={'trainer_source_sha256':SCRIPT,'model_contract_source_sha256':CONTRACT_SCRIPT,'threshold_free_policy_source_sha256':f'{REPO}/formal_NN_training/common/threshold_free_policy.py'}
SWEEP=[]
for point in POINTS:
    out=f"{LOCAL_OUTPUT}/{point['model_tag']}"; cmd=[sys.executable,SCRIPT,'--policy',POLICY]
    for role in ROLES: cmd += [f'--{role}-stream',INPUTS[role]['stream'],f'--{role}-candidates',INPUTS[role]['candidates']]
    cmd += ['--out-dir',out,'--model-family',point['model_family'],'--model-size',str(point['model_size']),'--pair-id',point['architecture_pair_id'],'--device','cuda','--seed',str(TRAINING['seed']),'--epochs',str(TRAINING['epochs']),'--chunk-len',str(TRAINING['chunk_len']),'--accumulate-chunks',str(TRAINING['accumulate_chunks']),'--learning-rate',str(TRAINING['learning_rate'])]
    print('\nTraining',point['model_tag'],' '.join(cmd),flush=True); subprocess.run(cmd,check=True)
    meta=json.loads(pathlib.Path(f'{out}/run_metadata.json').read_text())
    expected_meta={'model_tag':point['model_tag'],'model_family':point['model_family'],'model_size':point['model_size'],'architecture_pair_id':point['architecture_pair_id'],'matched_normal_prefetcher':POLICY,'same_external_input_contract':True,'training_inference_input_encoder_identical':True,'normal_policy_outputs_used_as_model_inputs':False,'normal_policy_candidates_used_as_model_inputs':False,'normal_policy_private_state_used_as_model_inputs':False,'normal_policy_outputs_used_as_training_targets':True,'normal_policy_templates_used_by_neural_inference':False,'probability_threshold_used':False,'threshold_related_hardcodes_used':False,'inference_policy_hardcodes_used':False,'neural_degree_cap':None,'same_page_rule_used_by_neural_inference':False}
    projected=('experiment_revision','model_revision','decoder_revision','runtime_feature_count','raw_runtime_feature_count','causal_runtime_feature_count','decoder_training_mode','delta_vocabulary_source','delta_vocabulary_max_exact','delta_other_escape','delta_other_decode_precision','exact_delta_representability_scope','rank_decision_training_objective','terminal_stop_supervised_for_every_teacher_sequence','checkpoint_selection','decode_per_callback_resource_watchdog','decode_per_role_resource_watchdog','decode_resource_watchdog_is_neural_degree_cap')
    expected_meta.update({key:MODEL_CONTRACT[key] for key in projected})
    bad={k:(meta.get(k),v) for k,v in expected_meta.items() if meta.get(k)!=v}; assert not bad,bad
    if 'run_id' in meta: assert meta['run_id']==RUN_ID
    assert meta['model_point_contract']==MODEL_CONTRACT and meta['decision_rule']==MODEL_CONTRACT['decoding_rule']
    assert meta['training_config']==TRAINING and meta['training_config_pinned_by_run_id'] is True
    assert meta['training_runtime_fields']==external_fields==meta['inference_runtime_fields']
    maximum=point['maximum_parameter_count']; assert 0<meta['parameter_count']==meta['realized_parameter_count']<=maximum
    assert meta['maximum_parameter_count']==maximum and meta['realized_parameter_count_matches_formula'] is True and meta['realized_parameter_count_within_maximum'] is True
    assert 0<meta['delta_vocabulary_exact_size']<=MODEL_CONTRACT['delta_vocabulary_max_exact']
    assert len(meta['delta_vocabulary_exact'])==meta['delta_vocabulary_exact_size']
    assert 1<=meta['selected_guard_epoch']<=TRAINING['epochs'] and meta['checkpoint_selection_roles']==['guard_metrics','TRAIN_loss_tiebreak_only']
    assert meta['guard_selection_composite_or_mean_used'] is False and len(meta['selected_guard_key'])==6
    determinism=MODEL_CONTRACT['determinism_contract']
    assert meta['training_device']=='cuda' and determinism['required_accelerator_name_contains'] in meta['training_device_name']
    assert meta['cublas_workspace_config']==determinism['cublas_workspace_config'] and meta['torch_deterministic_algorithms_enabled'] is True
    assert meta['cudnn_deterministic']==determinism['cudnn_deterministic'] and meta['cudnn_benchmark']==determinism['cudnn_benchmark'] and meta['float32_matmul_precision']==determinism['float32_matmul_precision']
    for key in MODEL_CONTRACT.get('required_source_hashes',source_paths):
        assert key in source_paths and meta[key]==hashlib.sha256(pathlib.Path(source_paths[key]).read_bytes()).hexdigest(),key
    encoder_hashes={meta.get('runtime_encoder_sha256'),meta.get('training_runtime_encoder_sha256'),meta.get('inference_runtime_encoder_sha256')}
    assert len(encoder_hashes)==1 and isinstance(next(iter(encoder_hashes)),str) and len(next(iter(encoder_hashes)))==64,encoder_hashes
    SWEEP.append({key:meta[key] for key in ('model_tag','model_family','model_size','architecture_pair_id','parameter_count','maximum_parameter_count','selected_guard_epoch','decision_rule','offline_normal_entries','offline_nn_entries','heldout_behavior_metrics')})
sweep_manifest={'run_id':RUN_ID,'trace':TRACE,'policy':POLICY,'model_revision':MODEL_CONTRACT['model_revision'],'decoder_revision':MODEL_CONTRACT['decoder_revision'],'training_config':TRAINING,'external_input_fields':external_fields,'input_archive':PART_MANIFEST['archive'],'model_contract':MODEL_CONTRACT,'points':SWEEP}
pathlib.Path(f'{LOCAL_OUTPUT}/sweep_manifest.json').write_text(json.dumps(sweep_manifest,indent=2,sort_keys=True)+'\n')
if os.path.isdir(OUTPUT_ROOT): shutil.rmtree(OUTPUT_ROOT)
shutil.copytree(LOCAL_OUTPUT,OUTPUT_ROOT)
print(json.dumps(SWEEP,indent=2))

In [ ]:
OUTPUT_ARCHIVE=pathlib.Path(DRIVE_ROOT,f'{RUN_ID}.colab_output.tar.gz')
with tarfile.open(OUTPUT_ARCHIVE,'w:gz') as archive:
    for item in sorted(pathlib.Path(OUTPUT_ROOT).iterdir(),key=lambda path:path.name): archive.add(item,arcname=item.name)
output_size=OUTPUT_ARCHIVE.stat().st_size
OUTPUT_PARTS_DIR=pathlib.Path(DRIVE_ROOT,'output_transfer_parts')
print('saved output in Drive',OUTPUT_ARCHIVE,output_size,'bytes')
if output_size<=transfer.MAX_PART_BYTES:
    if OUTPUT_PARTS_DIR.exists(): shutil.rmtree(OUTPUT_PARTS_DIR)
    print('Downloading one archive (<=90 MiB).')
    files.download(str(OUTPUT_ARCHIVE))
else:
    if OUTPUT_PARTS_DIR.exists(): shutil.rmtree(OUTPUT_PARTS_DIR)
    OUTPUT_PARTS_DIR.mkdir(parents=True)
    output_manifest_path=transfer.split_archive(OUTPUT_ARCHIVE,OUTPUT_PARTS_DIR,overwrite=True)
    output_manifest=transfer.validate_parts(output_manifest_path,OUTPUT_PARTS_DIR)
    assert output_manifest['archive']['sha256']==transfer.sha256_file(OUTPUT_ARCHIVE)
    print('Downloading multipart output (>90 MiB); keep the manifest and every part together.')
    files.download(str(output_manifest_path))
    for part in output_manifest['parts']: files.download(str(OUTPUT_PARTS_DIR/part['name']))

The complete output remains under the run-specific Google Drive folder. If the browser downloaded multipart output, retain its JSON manifest and all numbered parts; the same common utility verifies and rejoins them before the Sacramento replay stage.